In [ ]:
# Notebook 04: Basic RAG Pipeline

!pip install langchain langchain-community chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 17.6 MB/s eta 0:

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [ ]:


import chromadb
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from pathlib import Path

CHROMA_DIR = BASE_DIR / "data/vector_store"

print("RAG pipeline setup")


RAG pipeline setup


In [ ]:


# Initialize embeddings
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

# Load vector store
vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="arxiv_papers"
)

print(f"Vector store loaded: {vectorstore._collection.count()} documents")

Vector store loaded: 12728 documents


In [ ]:


# Create retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Maximum Marginal Relevance
    search_kwargs={
        'k': 5,
        'fetch_k': 20,
        'lambda_mult': 0.7
    }
)

print("Retriever configured")

Retriever configured


In [ ]:

# Test retrieval
test_queries = [
    "What are attention mechanisms in neural networks?",
    "How does BERT work?",
    "Explain reinforcement learning from human feedback",
    "What is the transformer architecture?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    docs = retriever.get_relevant_documents(query)
    print(f"Retrieved {len(docs)} documents")
    print(f"Top result: {docs[0].metadata['title']}")

print("\nNotebook 04 Complete!")


Query: What are attention mechanisms in neural networks?


/tmp/ipython-input-2515752856.py:12: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


Retrieved 5 documents
Top result: Decoupling Knowledge and Reasoning in Transformers: A Modular Architecture with Generalized Cross-Attention

Query: How does BERT work?
Retrieved 5 documents
Top result: The Text Classification Pipeline: Starting Shallow going Deeper

Query: Explain reinforcement learning from human feedback
Retrieved 5 documents
Top result: $β$-DQN: Improving Deep Q-Learning By Evolving the Behavior

Query: What is the transformer architecture?
Retrieved 5 documents
Top result: The Text Classification Pipeline: Starting Shallow going Deeper

Notebook 04 Complete!
